In [2]:
# ============================================================
# 病房感染監測儀表板（支援手動輸入檔名）
# ★ 只需修改 SEARCH_ROOT
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# ↓↓↓ 只需修改這一行 ↓↓↓
SEARCH_ROOT = '/content/drive/MyDrive/projects'
# ↑↑↑ 只需修改這一行 ↑↑↑

import pandas as pd
import re
import json
import os
import ipywidgets as widgets
from IPython.display import display, clear_output

def list_csv_files(folder_path):
    if not os.path.exists(folder_path):
        return []
    return sorted([
        os.path.join(folder_path, f)
        for f in os.listdir(folder_path)
        if f.lower().endswith('.csv')
    ])

def list_all_folders(base):
    if not os.path.exists(base):
        print(f"⚠️ 路徑不存在：{base}")
        return [base]
    folders = []
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        folders.append(root)
        if len(folders) > 200:
            break
    return sorted(folders)

all_folders = list_all_folders(base=SEARCH_ROOT)
print(f"✅ 找到 {len(all_folders)} 個資料夾（搜尋範圍：{SEARCH_ROOT}）")

# ── 步驟一：菌種 CSV ─────────────────────────────────────────
print("\n" + "="*55)
print("【步驟 1】選擇菌種資料夾與 CSV 檔")
print("="*55)
folder_selector_p = widgets.Dropdown(
    options=all_folders, description='資料夾：',
    layout=widgets.Layout(width='600px'),
    style={'description_width': '80px'}
)
file_selector_p = widgets.SelectMultiple(
    options=[], description='菌種CSV：',
    layout=widgets.Layout(width='600px', height='80px'),
    style={'description_width': '80px'}
)
def on_folder_change_p(change):
    file_selector_p.options = list_csv_files(change['new'])
folder_selector_p.observe(on_folder_change_p, names='value')
file_selector_p.options = list_csv_files(all_folders[0])
display(folder_selector_p, file_selector_p)

# ── 步驟二：抗生素 CSV ───────────────────────────────────────
print("\n" + "="*55)
print("【步驟 2】選擇抗生素資料夾與 CSV 檔（Ctrl 多選）")
print("="*55)
folder_selector_a = widgets.Dropdown(
    options=all_folders, description='資料夾：',
    layout=widgets.Layout(width='600px'),
    style={'description_width': '80px'}
)
file_selector_a = widgets.SelectMultiple(
    options=[], description='抗生素CSV：',
    layout=widgets.Layout(width='600px', height='160px'),
    style={'description_width': '80px'}
)
def on_folder_change_a(change):
    file_selector_a.options = list_csv_files(change['new'])
folder_selector_a.observe(on_folder_change_a, names='value')
file_selector_a.options = list_csv_files(all_folders[0])
display(folder_selector_a, file_selector_a)

# ── 步驟三：輸出資料夾 + 檔名 ───────────────────────────────
print("\n" + "="*55)
print("【步驟 3】選擇輸出資料夾與輸入檔名")
print("="*55)
folder_selector_out = widgets.Dropdown(
    options=all_folders, description='輸出資料夾：',
    layout=widgets.Layout(width='600px'),
    style={'description_width': '100px'}
)
# ★ 新增：手動輸入檔名
filename_input = widgets.Text(
    value='dashboard',
    description='檔案名稱：',
    placeholder='輸入檔名（不需加 .html）',
    layout=widgets.Layout(width='400px'),
    style={'description_width': '100px'}
)
display(folder_selector_out, filename_input)

# ── 步驟四：執行按鈕 ─────────────────────────────────────────
run_button = widgets.Button(
    description='▶ 產生儀表板',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px', margin='20px 0')
)
output_area = widgets.Output()
display(run_button, output_area)

def parse_antibiotic_csv(filepath):
    with open(filepath, 'r', encoding='utf-8-sig') as f:
        text = f.read()
    lines = text.splitlines()
    month_str = None
    for line in lines[:3]:
        m = re.search(r'(\d{4})(\d{2})\d{2}', line)
        if m:
            month_str = f"{m.group(1)}-{m.group(2)}"
            break
    if not month_str:
        fname = os.path.basename(filepath)
        m2 = re.search(r'(\d{1,2})', fname)
        month_num = int(m2.group(1)) if m2 else 0
        year = "2025" if month_num >= 10 else "2026"
        month_str = f"{year}-{month_num:02d}"
    from io import StringIO
    df = pd.read_csv(StringIO(text), skiprows=2, header=0)
    df.columns = [c.strip() for c in df.columns]
    df = df[df['ATC代碼'].notna() & (df['ATC代碼'].astype(str).str.strip() != '')].copy()
    df['ATC名稱'] = df['ATC名稱'].astype(str).str.strip()
    df['DDD總量'] = pd.to_numeric(df['DDD總量'], errors='coerce')
    df['住院天數'] = pd.to_numeric(df['住院天數'], errors='coerce')
    bed_days = df['住院天數'].dropna().iloc[0]
    df['月份'] = month_str
    df['DDD_per_day'] = (df['DDD總量'] / bed_days).round(4)
    return df[['月份', 'ATC代碼', 'ATC名稱', 'DDD總量', 'DDD_per_day']], month_str, bed_days

def on_run_clicked(b):
    with output_area:
        clear_output()
        if not file_selector_p.value:
            print("❌ 請先選擇菌種 CSV 檔案")
            return
        if not file_selector_a.value:
            print("❌ 請先選擇抗生素 CSV 檔案")
            return

        df_pathogen = pd.read_csv(file_selector_p.value[0], encoding='utf-8-sig')
        df_pathogen.columns = ['年月', '菌種', '株數']
        df_pathogen['年月'] = df_pathogen['年月'].astype(str).str.strip()
        df_pathogen['菌種'] = df_pathogen['菌種'].str.strip()
        print(f"✅ 菌種資料：{len(df_pathogen)} 筆，月份：{sorted(df_pathogen['年月'].unique())}")

        abx_dfs, bed_days_map = [], {}
        for fpath in file_selector_a.value:
            df_parsed, month_str, bed_days = parse_antibiotic_csv(fpath)
            abx_dfs.append(df_parsed)
            bed_days_map[month_str] = bed_days
            print(f"✅ {os.path.basename(fpath)}：月份={month_str}，住院天數={int(bed_days)}")

        df_abx = pd.concat(abx_dfs, ignore_index=True).sort_values('月份')
        all_months = sorted(set(df_pathogen['年月'].unique()) | set(df_abx['月份'].unique()))

        df_p = df_pathogen.copy()
        df_p['住院天數'] = df_p['年月'].map(bed_days_map)
        avg_bed = sum(bed_days_map.values()) / len(bed_days_map)
        df_p['住院天數'] = df_p['住院天數'].fillna(avg_bed)
        df_p['株數_per_1000'] = (df_p['株數'] / df_p['住院天數'] * 1000).round(2)

        pivot_p = (df_p.pivot_table(index='年月', columns='菌種', values='株數_per_1000', aggfunc='sum')
                   .reindex(all_months).fillna(0))
        pathogens = pivot_p.columns.tolist()

        pivot_a = (df_abx.pivot_table(index='月份', columns='ATC名稱', values='DDD_per_day', aggfunc='sum')
                   .reindex(all_months).fillna(0))
        drugs = pivot_a.columns.tolist()

        print(f"\n涵蓋月份：{all_months}")
        print(f"菌種：{len(pathogens)} 種 | 藥物：{len(drugs)} 種")

        def make_traces(pivot_df, names, hover_label, decimals):
            traces = []
            for name in names:
                y_vals = [round(float(pivot_df.loc[m, name]), decimals)
                          if m in pivot_df.index else 0 for m in all_months]
                traces.append({
                    "name": name, "x": all_months, "y": y_vals,
                    "mode": "lines+markers", "type": "scatter",
                    "hovertemplate": f"<b>{name}</b><br>月份: %{{x}}<br>{hover_label}: %{{y}}<extra></extra>"
                })
            return traces

        traces_p = make_traces(pivot_p, pathogens, "株數/千住院日", 2)
        traces_a = make_traces(pivot_a, drugs, "DDD/住院日", 4)
        tj_p = json.dumps(traces_p, ensure_ascii=False)
        tj_a = json.dumps(traces_a, ensure_ascii=False)
        pj   = json.dumps(pathogens, ensure_ascii=False)
        dj   = json.dumps(drugs, ensure_ascii=False)
        mj   = json.dumps(all_months)

        html_content = f"""<!DOCTYPE html>
<html lang="zh-TW">
<head>
<meta charset="UTF-8">
<title>病房感染監測儀表板</title>
<script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
<style>
* {{ box-sizing: border-box; margin: 0; padding: 0; }}
body {{ font-family: 'Segoe UI', Arial, sans-serif; background: #f5f7fa;
        display: flex; height: 100vh; overflow: hidden; }}
#sidebar {{ width: 260px; min-width: 220px; background: #fff;
  border-right: 1px solid #e0e0e0; display: flex; flex-direction: column; overflow: hidden; }}
.panel {{ flex: 1; display: flex; flex-direction: column; padding: 12px;
  overflow: hidden; border-bottom: 2px solid #e8ecf0; }}
.panel:last-child {{ border-bottom: none; }}
.panel h3 {{ font-size: 12px; font-weight: 700; color: #444; margin-bottom: 6px; }}
.btn-row {{ display: flex; gap: 5px; margin-bottom: 8px; }}
.btn-row button {{ flex: 1; padding: 4px 0; font-size: 11px; border: 1px solid #ccc;
  border-radius: 4px; cursor: pointer; background: #f0f0f0; }}
.btn-row button:hover {{ background: #ddd; }}
.item-list {{ overflow-y: auto; flex: 1; }}
.item {{ display: flex; align-items: flex-start; gap: 7px; padding: 3px 4px;
  border-radius: 3px; font-size: 11px; color: #333; }}
.item:hover {{ background: #f0f4ff; }}
.item input {{ cursor: pointer; margin-top: 2px; accent-color: #4a90d9; flex-shrink: 0; }}
.dot {{ width: 10px; height: 10px; border-radius: 50%; flex-shrink: 0; margin-top: 3px; }}
.item label {{ cursor: pointer; line-height: 1.35; word-break: break-word; }}
#charts {{ flex: 1; display: flex; flex-direction: column; padding: 10px; gap: 10px; overflow: hidden; }}
.chart-wrap {{ flex: 1; background: #fff; border-radius: 8px;
  box-shadow: 0 1px 4px rgba(0,0,0,0.08); overflow: hidden; }}
</style>
</head>
<body>
<div id="sidebar">
  <div class="panel">
    <h3>🦠 菌種（株數／千住院日）</h3>
    <div class="btn-row">
      <button onclick="selectAll('p')">全選</button>
      <button onclick="selectNone('p')">全消</button>
    </div>
    <div class="item-list" id="list_p"></div>
  </div>
  <div class="panel">
    <h3>💊 抗生素（DDD／住院日）</h3>
    <div class="btn-row">
      <button onclick="selectAll('a')">全選</button>
      <button onclick="selectNone('a')">全消</button>
    </div>
    <div class="item-list" id="list_a"></div>
  </div>
</div>
<div id="charts">
  <div class="chart-wrap"><div id="chart_p" style="width:100%;height:100%"></div></div>
  <div class="chart-wrap"><div id="chart_a" style="width:100%;height:100%"></div></div>
</div>
<script>
const MONTHS={mj},traces_p={tj_p},traces_a={tj_a},pathogens={pj},drugs={dj};
const COLORS=['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b',
  '#e377c2','#7f7f7f','#bcbd22','#17becf','#aec7e8','#ffbb78','#98df8a',
  '#ff9896','#c5b0d5','#c49c94','#f7b6d2','#c7c7c7','#dbdb8d','#9edae5',
  '#393b79','#637939','#8c6d31','#843c39','#7b4173'];
const lb={{xaxis:{{title:'月份',type:'category',tickangle:-20}},hovermode:'x unified',
  plot_bgcolor:'#fff',paper_bgcolor:'#fff',margin:{{t:45,l:70,r:20,b:50}},
  legend:{{orientation:'h',y:-0.25}}}};
Plotly.newPlot('chart_p',traces_p.map((t,i)=>
  ({{...t,line:{{color:COLORS[i%COLORS.length],width:2}},marker:{{size:6}}}})),
  {{...lb,title:{{text:'菌種株數趨勢（株數 / 千住院日）',font:{{size:15}}}},
    yaxis:{{title:'株數 / 千住院日',rangemode:'tozero'}}}},{{responsive:true}});
Plotly.newPlot('chart_a',traces_a.map((t,i)=>
  ({{...t,line:{{color:COLORS[i%COLORS.length],width:2}},marker:{{size:6}}}})),
  {{...lb,title:{{text:'抗生素 DDD / 住院日數趨勢',font:{{size:15}}}},
    yaxis:{{title:'DDD / 住院日數',rangemode:'tozero'}}}},{{responsive:true}});
function buildList(listId,names,chartId){{
  const c=document.getElementById(listId);
  names.forEach((name,i)=>{{
    const color=COLORS[i%COLORS.length],div=document.createElement('div');
    div.className='item';
    div.innerHTML=`<input type="checkbox" id="${{listId}}_${{i}}" checked
      onchange="updateChart('${{listId}}',${{JSON.stringify(names)}},'${{chartId}}')">
      <span class="dot" style="background:${{color}}"></span>
      <label for="${{listId}}_${{i}}">${{name}}</label>`;
    c.appendChild(div);
  }});
}}
buildList('list_p',pathogens,'chart_p');
buildList('list_a',drugs,'chart_a');
function updateChart(listId,names,chartId){{
  Plotly.restyle(chartId,'visible',names.map((_,i)=>
    document.getElementById(listId+'_'+i).checked?true:'legendonly'));
}}
function selectAll(type){{
  const l='list_'+type,names=type==='p'?pathogens:drugs;
  names.forEach((_,i)=>document.getElementById(l+'_'+i).checked=true);
  updateChart(l,names,'chart_'+type);
}}
function selectNone(type){{
  const l='list_'+type,names=type==='p'?pathogens:drugs;
  names.forEach((_,i)=>document.getElementById(l+'_'+i).checked=false);
  updateChart(l,names,'chart_'+type);
}}
</script>
</body>
</html>"""

        # ★ 使用手動輸入的檔名
        fname_input = filename_input.value.strip() or 'dashboard'
        if not fname_input.endswith('.html'):
            fname_input += '.html'

        out_folder = folder_selector_out.value
        os.makedirs(out_folder, exist_ok=True)
        out_path = os.path.join(out_folder, fname_input)
        with open(out_path, 'w', encoding='utf-8') as f:
            f.write(html_content)
        print(f"\n✅ 儀表板已儲存至：{out_path}")

run_button.on_click(on_run_clicked)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 找到 8 個資料夾（搜尋範圍：/content/drive/MyDrive/projects）

【步驟 1】選擇菌種資料夾與 CSV 檔


Dropdown(description='資料夾：', layout=Layout(width='600px'), options=('/content/drive/MyDrive/projects', '/conte…

SelectMultiple(description='菌種CSV：', layout=Layout(height='80px', width='600px'), options=(), style=Descriptio…


【步驟 2】選擇抗生素資料夾與 CSV 檔（Ctrl 多選）


Dropdown(description='資料夾：', layout=Layout(width='600px'), options=('/content/drive/MyDrive/projects', '/conte…

...


【步驟 3】選擇輸出資料夾與輸入檔名


Dropdown(description='輸出資料夾：', layout=Layout(width='600px'), options=('/content/drive/MyDrive/projects', '/con…

Text(value='dashboard', description='檔案名稱：', layout=Layout(width='400px'), placeholder='輸入檔名（不需加 .html）', styl…

Button(button_style='success', description='▶ 產生儀表板', layout=Layout(height='40px', margin='20px 0', width='200…

Output()